[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch06-nn-architectures/05-ben-trong-co-che-tap-trung.ipynb)

# Bên trong cơ chế tập trung: truy vấn, vị trí, mặt nạ và nhiều đầu

Sổ tay 03 đã tính cái giá của cơ chế tập trung (attention mechanism): ma trận điểm bậc hai, bộ nhớ đệm KV, bức
tường băng thông. Sổ tay này mở nó ra xem bên trong: mỗi phần của công thức
$\text{softmax}(QK^\top/\sqrt{d_k})\,V$ làm gì, cái gì hỏng khi bỏ nó đi, và một transformer nhỏ học được gì khi
huấn luyện ngay trên CPU.

**Bạn sẽ làm:**
1. viết cơ chế tập trung bằng tay như một phép tra cứu mềm, rồi kiểm với `F.scaled_dot_product_attention`;
2. đo vì sao phải chia cho $\sqrt{d_k}$: softmax bão hoà khi đầu rộng ra, và tràn trên ở FP16;
3. xem mặt nạ nhân quả (causal mask) chặn điều gì, và một mô hình học gian lận khi thiếu nó;
4. kiểm rằng không có thông tin vị trí thì tự tập trung không phân biệt được thứ tự, rồi so mã hoá vị trí hình sin
   với mã hoá vị trí xoay (RoPE);
5. đếm cái giá của nhiều đầu;
6. huấn luyện một transformer hai tầng đảo ngược chuỗi, xem từng đầu nhìn vào đâu, rồi thử nó trên chuỗi dài hơn
   lúc huấn luyện.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a5cc2"]
FP16 = 2
print("sẵn sàng")

## 1. Tập trung là một phép tra cứu mềm

Chương ví cơ chế tập trung với một bảng băm: truy vấn được so với mọi khoá, rồi kết quả là một tổ hợp có trọng số
của các giá trị. Khác bảng băm ở chỗ không có khoá nào "trúng" hay "trượt" hẳn: tích vô hướng $q \cdot k_j$ đo
mức khớp, và softmax biến các mức khớp thành trọng số có tổng bằng một.

Ô dưới dựng một "từ điển" sáu khoá trực chuẩn, mỗi khoá 16 chiều, kèm sáu giá trị. Truy vấn là khoá số 2 nhân với
một hệ số $\beta$, cộng một chút nhiễu.

**Dự đoán:** khi $\beta$ tăng, trọng số dồn về đâu? Đầu ra tiến về vectơ nào? Tổng các trọng số có đổi không?

In [ ]:
dat_hat_giong()
d_tu_dien = 16
khoa = torch.linalg.qr(torch.randn(d_tu_dien, 6)).Q.T     # 6 khoá trực chuẩn, mỗi khoá một hàng
gia_tri = torch.randn(6, 4)                                # 6 giá trị, mỗi giá trị 4 chiều
nhieu = 0.1 * torch.randn(d_tu_dien)
for beta in (1, 8, 32):
    q = beta * khoa[2] + nhieu
    w = torch.softmax(q @ khoa.T / math.sqrt(d_tu_dien), dim=-1)
    o = w @ gia_tri
    print(f"β = {beta:>2}: trọng số {np.array2string(w.numpy(), precision=3, suppress_small=True)}"
          f"  tổng = {w.sum():.6f}  khoảng cách từ đầu ra tới giá trị số 2 = {(o - gia_tri[2]).norm():.3f}")

Với $\beta$ nhỏ, đầu ra là một trung bình nhoè của mọi giá trị. Với $\beta$ lớn, gần như toàn bộ trọng số rơi vào
khoá khớp nhất, và đầu ra gần như đúng bằng giá trị của nó: phép tra cứu mềm trở thành một phép tra cứu cứng. Tổng
luôn bằng một. Chính vì mềm nên nó khả vi, và gradient chảy được về cả truy vấn lẫn khoá.

Trong tự tập trung (self-attention), cả truy vấn, khoá và giá trị đều là phép chiếu của cùng một chuỗi đầu vào $X$
(phương trình 7 của chương):

$$\text{SelfAttention}(X) = \text{softmax}\left(\frac{XW_Q (XW_K)^\top}{\sqrt{d_k}}\right) XW_V .$$

Ô dưới viết đúng công thức đó bằng ba phép nhân ma trận và một softmax, rồi so với hàm có sẵn của PyTorch.

In [ ]:
dat_hat_giong()
S_vd, d_vd = 10, 32
X = torch.randn(S_vd, d_vd)
W_Q, W_K, W_V = (torch.randn(d_vd, d_vd) / math.sqrt(d_vd) for _ in range(3))
Q, K, V = X @ W_Q, X @ W_K, X @ W_V
trong_so = torch.softmax(Q @ K.T / math.sqrt(d_vd), dim=-1)
bang_tay = trong_so @ V
co_san = F.scaled_dot_product_attention(Q[None], K[None], V[None])[0]
assert torch.allclose(bang_tay, co_san, atol=1e-5)
print(f"bằng tay khớp F.scaled_dot_product_attention: lệch lớn nhất {(bang_tay - co_san).abs().max():.1e}")
print(f"ma trận trọng số {tuple(trong_so.shape)}: mỗi hàng cộng lại bằng một, lệch lớn nhất "
      f"{(trong_so.sum(-1) - 1).abs().max():.1e}")

## 2. Vì sao chia cho $\sqrt{d_k}$

Chương giải thích: nếu các thành phần của truy vấn và khoá độc lập, trung bình 0 và phương sai 1, thì tích vô hướng
của chúng có phương sai $d_k$, và chia cho $\sqrt{d_k}$ đưa phương sai về 1, giữ cho gradient ổn định. Ô dưới đo
điều đó: lấy 256 truy vấn và 128 khoá ngẫu nhiên, tính điểm có chia và không chia, rồi xem mỗi hàng softmax trông
thế nào. Hai thước đo: trọng số lớn nhất của một hàng (1/128 nếu đều tuyệt đối, 1 nếu dồn hết vào một khoá), và
entropy của hàng chia cho entropy lớn nhất có thể (1 nếu đều, 0 nếu chỉ một khoá).

**Dự đoán:** với đầu 1,024 chiều và không chia, trọng số lớn nhất của một hàng khoảng bao nhiêu?

In [ ]:
cac_d = [4, 16, 64, 128, 256, 1024]
N_KHOA = 128
ket_qua = {"không chia": [], "chia cho √d": []}
g = torch.Generator().manual_seed(SEED)
for d in cac_d:
    q, k = torch.randn(256, d, generator=g), torch.randn(N_KHOA, d, generator=g)
    diem = q @ k.T
    for ten, s in (("không chia", diem), ("chia cho √d", diem / math.sqrt(d))):
        p = torch.softmax(s, dim=-1)
        entropy = -(p * p.clamp_min(1e-30).log()).sum(-1).mean().item()
        ket_qua[ten].append((s.var().item(), p.max(-1).values.mean().item(), entropy / math.log(N_KHOA)))
print(f"{'d':>6} | {'phương sai điểm':>24} | {'trọng số lớn nhất':>24} | {'entropy / entropy tối đa':>24}")
print(f"{'':>6} | {'không chia':>11} {'chia √d':>12} | {'không chia':>11} {'chia √d':>12} | {'không chia':>11} {'chia √d':>12}")
for i, d in enumerate(cac_d):
    a, b = ket_qua["không chia"][i], ket_qua["chia cho √d"][i]
    print(f"{d:>6} | {a[0]:>11.1f} {b[0]:>12.2f} | {a[1]:>11.3f} {b[1]:>12.3f} | {a[2]:>11.2f} {b[2]:>12.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.6, 3.6), sharex=True)
for j, (ten, mau) in enumerate((("không chia", MAU[1]), ("chia cho √d", MAU[0]))):
    ax1.plot(cac_d, [r[1] for r in ket_qua[ten]], "o-", color=mau, label=ten)
    ax2.plot(cac_d, [r[2] for r in ket_qua[ten]], "o-", color=mau, label=ten)
ax1.axhline(1 / N_KHOA, color="0.5", lw=0.8, ls="--", label="đều tuyệt đối: 1/128")
for ax in (ax1, ax2):
    ax.set_xscale("log", base=2)
    ax.set_xticks(cac_d, [str(d) for d in cac_d])
    ax.minorticks_off()
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("số chiều của đầu d")
ax1.set_ylabel("trọng số lớn nhất (trung bình hàng)")
ax2.set_ylabel("entropy / entropy tối đa")
ax1.set_title("Một khoá chiếm hết?", fontsize=10)
ax2.set_title("Hàng softmax còn trải rộng?", fontsize=10)
ax1.legend(loc="center right", fontsize=8)
plt.tight_layout()
plt.show()

Không chia, phương sai của điểm bằng đúng $d$, nên đầu càng rộng thì điểm càng xa nhau, và softmax càng dồn gần hết
trọng số vào một khoá. Một hàng softmax bão hoà như thế có đạo hàm gần bằng 0 theo mọi điểm của nó ($p(1-p)$ gần 0
khi $p$ gần 0 hoặc 1), nên gradient gần như không chảy về truy vấn và khoá: mô hình khó học. Chia cho $\sqrt{d}$,
cả ba thước đo đứng yên khi $d$ đổi.

### Softmax và độ chính xác thấp

Chương nói thêm rằng phép softmax nhạy với độ chính xác thấp. Số lớn nhất FP16 biểu diễn được là 65,504, nên
$e^x$ tràn trên khi $x$ vượt quá chừng 11. Với đầu 128 chiều như ví dụ của chương, điểm không chia có độ lệch
chuẩn $\sqrt{128}$.

**Dự đoán:** nếu tính softmax ngây thơ ($e^x$ rồi chia cho tổng) ở FP16, bao nhiêu phần trăm số hàng hỏng?

In [ ]:
D_DAU = sach(128, trich="128-dimensional heads")
FP16_MAX = torch.finfo(torch.float16).max
print(f"số lớn nhất FP16: {FP16_MAX:,.0f}, nên exp(x) tràn trên khi x > ln({FP16_MAX:,.0f}) = {math.log(FP16_MAX):.2f}")
print(f"điểm không chia, đầu {D_DAU} chiều: độ lệch chuẩn √{D_DAU} = {math.sqrt(D_DAU):.2f}")
g = torch.Generator().manual_seed(SEED)
q, k = torch.randn(1000, D_DAU, generator=g), torch.randn(N_KHOA, D_DAU, generator=g)


def softmax_ngay_tho(s):
    e = torch.exp(s)
    return e / e.sum(-1, keepdim=True)


def softmax_on_dinh(s):
    e = torch.exp(s - s.max(-1, keepdim=True).values)   # trừ giá trị lớn nhất: mọi số mũ ≤ 0
    return e / e.sum(-1, keepdim=True)


for ten, s in (("không chia", q @ k.T), ("chia cho √d", q @ k.T / math.sqrt(D_DAU))):
    chuan = torch.softmax(s.double(), dim=-1)
    for cach, f in (("ngây thơ", softmax_ngay_tho), ("trừ giá trị lớn nhất", softmax_on_dinh)):
        p = f(s.half())
        hong = (~torch.isfinite(p)).any(-1).float().mean().item()
        lech = (p.double() - chuan).abs().nan_to_num(nan=1.0).max().item()
        print(f"{ten:>12}, FP16, {cach:<21}: {100 * hong:5.1f} % số hàng có inf/NaN; lệch lớn nhất so với FP64 {lech:.1e}")

Ở FP16, softmax ngây thơ trên điểm không chia hỏng gần như mọi hàng: chỉ cần một điểm vượt chừng 11 là $e^x$ thành
vô cực, và vô cực chia cho vô cực là NaN. Trừ giá trị lớn nhất của hàng trước khi lấy mũ không đổi kết quả về mặt
toán (nó triệt tiêu ở tử và mẫu), nhưng đưa mọi số mũ về $\le 0$, nên không còn gì tràn. `torch.softmax` và mọi
kernel tập trung đều làm việc này bên trong. Softmax trực tuyến (online softmax) ở sổ tay 03 cũng chính là nó, chỉ
là áp dụng từng khối một.

## 3. Mặt nạ nhân quả: không được nhìn vào tương lai

Khi sinh token theo kiểu tự hồi quy (autoregressive), token ở vị trí $t$ phải được đoán chỉ từ các token trước nó.
Nhưng khi huấn luyện, cả chuỗi được đưa vào cùng lúc để mọi vị trí được tính song song. Mặt nạ nhân quả nối hai điều
đó: nó đặt điểm $-\infty$ cho mọi cặp (truy vấn $i$, khoá $j$) với $j > i$, nên sau softmax các vị trí phía sau có
trọng số đúng bằng 0.

Ô dưới dựng mô hình dùng cho cả sổ tay: embedding, các khối transformer chuẩn hoá trước (pre-LN), và một tầng ra. Nó
có hai lựa chọn: có mặt nạ nhân quả hay không, và mã hoá vị trí kiểu gì (phần 4 sẽ nói kỹ). Nó giữ lại ma trận
trọng số tập trung của lượt chạy gần nhất để vẽ ở phần 6.

In [ ]:
def hinh_sin(T, d):
    """Mã hoá vị trí hình sin của Vaswani và cộng sự: cộng vào embedding trước tầng đầu tiên."""
    vi = torch.arange(T, dtype=torch.float32)[:, None]
    tan_so = 10000 ** (-torch.arange(0, d, 2, dtype=torch.float32) / d)
    pe = torch.zeros(T, d)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(vi * tan_so), torch.cos(vi * tan_so)
    return pe


def xoay(x, co_so=10000.0):
    """RoPE: quay cặp toạ độ (2i, 2i+1) của vectơ ở vị trí m một góc m·θ_i, θ_i = co_so^(−2i/d)."""
    T, d = x.shape[-2], x.shape[-1]
    theta = co_so ** (-torch.arange(0, d, 2, dtype=x.dtype) / d)
    goc = torch.arange(T, dtype=x.dtype)[:, None] * theta
    c, s = goc.cos(), goc.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack((x1 * c - x2 * s, x1 * s + x2 * c), -1).flatten(-2)


class TapTrung(nn.Module):
    def __init__(self, d, dau, vi_tri, nhan_qua):
        super().__init__()
        self.dau, self.d_dau, self.vi_tri, self.nhan_qua = dau, d // dau, vi_tri, nhan_qua
        self.qkv = nn.Linear(d, 3 * d)
        self.ra = nn.Linear(d, d)

    def forward(self, x):                                   # x: (B, T, d)
        B, T, _ = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.dau, self.d_dau).permute(2, 0, 3, 1, 4)
        if self.vi_tri == "rope":
            q, k = xoay(q), xoay(k)
        diem = q @ k.transpose(-1, -2) / math.sqrt(self.d_dau)  # (B, đầu, T, T)
        if self.nhan_qua:
            tuong_lai = torch.ones(T, T, dtype=torch.bool).triu(1)
            diem = diem.masked_fill(tuong_lai, -math.inf)
        w = torch.softmax(diem, dim=-1)
        self.trong_so = w.detach()
        return self.ra((w @ v).transpose(1, 2).reshape(B, T, -1))


class Khoi(nn.Module):
    def __init__(self, d, dau, d_ff, vi_tri, nhan_qua):
        super().__init__()
        self.chuan1, self.chuan2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.tap_trung = TapTrung(d, dau, vi_tri, nhan_qua)
        self.truyen_thang = nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Linear(d_ff, d))

    def forward(self, x):
        x = x + self.tap_trung(self.chuan1(x))
        return x + self.truyen_thang(self.chuan2(x))


class MoHinh(nn.Module):
    def __init__(self, V=16, d=64, dau=4, tang=2, d_ff=128, vi_tri="sin", nhan_qua=False):
        super().__init__()
        self.V, self.d, self.vi_tri = V, d, vi_tri
        self.embedding = nn.Embedding(V, d)
        self.khoi = nn.ModuleList(Khoi(d, dau, d_ff, vi_tri, nhan_qua) for _ in range(tang))
        self.chuan = nn.LayerNorm(d)
        self.ra = nn.Linear(d, V)

    def forward(self, x):                                   # x: (B, T) chỉ số token
        h = self.embedding(x)
        if self.vi_tri == "sin":
            h = h + hinh_sin(x.shape[1], self.d)
        for khoi in self.khoi:
            h = khoi(h)
        return self.ra(self.chuan(h))


def do_chinh_xac(mo_hinh, x, y):
    """Tỉ lệ token đoán đúng, chỉ ở những vị trí có đích (đích −100 nghĩa là không chấm)."""
    with torch.no_grad():
        du_doan = mo_hinh(x).argmax(-1)
    chon = y != -100
    return (du_doan[chon] == y[chon]).float().mean().item()


def huan_luyen(mo_hinh, bai, buoc, kiem_tra, moi=100, lr=3e-3, batch=64):
    """Adam, entropy chéo trên các vị trí có đích. Trả về độ chính xác trên tập kiểm tra sau mỗi `moi` bước."""
    opt = torch.optim.Adam(mo_hinh.parameters(), lr=lr, foreach=False)
    g = torch.Generator().manual_seed(0)                    # cùng một dòng batch cho mọi biến thể
    duong, mat_mat = [], []
    t0 = time.perf_counter()
    for b in range(1, buoc + 1):
        x, y = bai(batch, g)
        loss = F.cross_entropy(mo_hinh(x).reshape(-1, mo_hinh.V), y.reshape(-1), ignore_index=-100)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if b % moi == 0:
            duong.append(do_chinh_xac(mo_hinh, *kiem_tra))
            mat_mat.append(loss.item())
    return duong, mat_mat, time.perf_counter() - t0


print(f"mô hình mặc định: {sum(p.numel() for p in MoHinh().parameters()):,} tham số")

Phép thử đầu tiên không cần huấn luyện. Lấy một chuỗi 16 token, đổi đúng token ở vị trí 10, rồi xem đầu ra ở những
vị trí nào thay đổi.

**Dự đoán:** không có mặt nạ, đổi token 10 có làm đổi đầu ra ở vị trí 3 không? Có mặt nạ thì sao?

In [ ]:
x = torch.randint(0, 16, (1, 16), generator=torch.Generator().manual_seed(3))
x_doi = x.clone()
x_doi[0, 10] = (x_doi[0, 10] + 1) % 16
for nhan_qua in (False, True):
    dat_hat_giong()
    m = MoHinh(nhan_qua=nhan_qua).eval()
    with torch.no_grad():
        doi = (m(x) - m(x_doi)).abs().amax(-1)[0]
    print(f"{'có' if nhan_qua else 'không có'} mặt nạ: thay đổi lớn nhất ở các vị trí 0–9 = {doi[:10].max():.2e}, "
          f"nhỏ nhất ở các vị trí 10–15 = {doi[10:].min():.2e}")

Có mặt nạ, các vị trí đứng trước token bị đổi không nhận được một bit thông tin nào từ nó: thay đổi đúng bằng 0, chứ
không chỉ nhỏ. Không có mặt nạ, mọi vị trí đều đổi.

Giờ xem điều đó quan trọng thế nào khi huấn luyện. Lấy các chuỗi 17 token **hoàn toàn ngẫu nhiên** trong 16 từ, và
huấn luyện mô hình đoán token kế tiếp ở mỗi vị trí. Không có cách trung thực nào đoán được một token ngẫu nhiên, nên
mất mát entropy chéo tốt nhất có thể là $\ln 16$.

**Dự đoán:** mô hình không có mặt nạ sẽ đạt mất mát bao nhiêu? Và nó đoán đúng bao nhiêu khi phải sinh token thật,
lúc token kế tiếp chưa tồn tại?

In [ ]:
def bai_ngau_nhien(B, g, T=16):
    x = torch.randint(0, 16, (B, T + 1), generator=g)
    y = torch.cat([x[:, 1:], torch.full((B, 1), -100)], 1)      # đích ở vị trí i là token i + 1
    return x, y


kiem_tra_nn = bai_ngau_nhien(500, torch.Generator().manual_seed(1))
duong_mm = {}
for nhan_qua in (True, False):
    dat_hat_giong()
    m = MoHinh(nhan_qua=nhan_qua)
    _, mat_mat, giay = huan_luyen(m, bai_ngau_nhien, 300, kiem_tra_nn, moi=25)
    duong_mm[nhan_qua] = mat_mat
    x_kt = kiem_tra_nn[0]
    thay_ca_chuoi = do_chinh_xac(m, *kiem_tra_nn)
    with torch.no_grad():                                   # sinh thật: chỉ đưa vào các token đã có
        that = np.mean([(m(x_kt[:, :i + 1])[:, -1].argmax(-1) == x_kt[:, i + 1]).float().mean().item()
                        for i in range(16)])
    ten = "có mặt nạ" if nhan_qua else "không có mặt nạ"
    do_tren_may(f"{ten}: mất mát entropy chéo cuối cùng", round(mat_mat[-1], 3))
    do_tren_may(f"{ten}: đoán đúng khi thấy cả chuỗi", round(thay_ca_chuoi, 3))
    do_tren_may(f"{ten}: đoán đúng khi token kế tiếp chưa có", round(that, 3))
    do_tren_may(f"{ten}: thời gian huấn luyện", round(giay, 1), "s")
print(f"ln 16 = {math.log(16):.3f}; đoán mò đúng 1/16 = {1 / 16:.4f}")

fig, ax = plt.subplots(figsize=(6.8, 3.6))
buoc_ve = np.arange(25, 301, 25)
ax.plot(buoc_ve, duong_mm[True], "o-", color=MAU[0], label="có mặt nạ nhân quả")
ax.plot(buoc_ve, duong_mm[False], "o-", color=MAU[1], label="không có mặt nạ")
ax.axhline(math.log(16), color="0.5", lw=0.8, ls="--", label="ln 16: tốt nhất có thể nếu không nhìn trộm")
ax.set_ylim(-0.1, 3.2)
ax.set_xlabel("bước huấn luyện")
ax.set_ylabel("mất mát entropy chéo")
ax.set_title("Đoán token kế tiếp của chuỗi ngẫu nhiên (đo trên máy này)")
ax.legend(loc="center right", fontsize=8)
plt.tight_layout()
plt.show()

Mô hình không có mặt nạ đẩy mất mát xuống dưới mức không thể, vì nó không đoán: nó đọc đáp án ở vị trí kế bên. Khi
thấy cả chuỗi, nó đúng gần như tuyệt đối. Khi phải sinh token thật, đáp án chưa tồn tại, và nó rơi về mức đoán mò.
Một con số huấn luyện đẹp như thế là dấu hiệu rò rỉ, không phải dấu hiệu học tốt. Mô hình có mặt nạ đứng đúng ở
$\ln 16$, mức trung thực.

## 4. Thứ tự: không có vị trí, tự tập trung chỉ thấy một tập hợp

Chương nói tự tập trung cho mọi vị trí tương tác với mọi vị trí, còn thứ tự của chuỗi phải do các cơ chế vị trí mã
hoá riêng. Lý do: nếu đổi chỗ các token đầu vào, mọi truy vấn, khoá và giá trị đổi chỗ theo, và tổ hợp có trọng số
của mỗi hàng giữ nguyên. Nói cách khác, tự tập trung không có vị trí là đẳng biến với hoán vị (permutation
equivariant): hoán vị đầu vào thì đầu ra hoán vị y như thế, không hơn không kém.

In [ ]:
x = torch.randint(0, 16, (1, 16), generator=torch.Generator().manual_seed(5))
hoan_vi = torch.randperm(16, generator=torch.Generator().manual_seed(6))
for vi_tri in ("khong", "sin", "rope"):
    dat_hat_giong()
    m = MoHinh(vi_tri=vi_tri).eval()
    with torch.no_grad():
        lech = (m(x[:, hoan_vi]) - m(x)[:, hoan_vi]).abs().max().item()
    print(f"vị trí '{vi_tri}': |M(x hoán vị) − M(x) hoán vị| lớn nhất = {lech:.1e}"
          f"  -> {'đẳng biến: thứ tự vô hình' if lech < 1e-5 else 'thấy được thứ tự'}")

Có hai cách phổ biến để đưa vị trí vào.

* **Mã hoá vị trí hình sin** (positional encoding, bài báo transformer gốc): cộng vào embedding của token ở vị trí $m$
  một vectơ cố định gồm các hàm $\sin$ và $\cos$ của $m$ ở nhiều tần số. Vị trí tuyệt đối đi vào nội dung.
* **Mã hoá vị trí xoay** (rotary position embedding, RoPE): không cộng gì vào embedding. Ngay trước khi tính điểm,
  mỗi cặp toạ độ của truy vấn và khoá ở vị trí $m$ được quay một góc $m\theta_i$. Vì tích vô hướng của hai vectơ bị
  quay chỉ phụ thuộc hiệu hai góc, điểm giữa vị trí $m$ và vị trí $n$ chỉ phụ thuộc $m - n$.

Chương chỉ nêu tên cơ chế vị trí; RoPE và mọi phép đo về nó ở đây là của sổ tay này, không phải của sách. Ô dưới
lấy **một** truy vấn và **một** khoá cố định, đặt chúng vào mọi cặp vị trí $(m, n)$ trong 32 vị trí, rồi vẽ điểm
$q_m \cdot k_n$ theo cả hai cách.

**Dự đoán:** với RoPE, các ô nằm trên cùng một đường chéo (cùng $m - n$) có bằng nhau không?

In [ ]:
dat_hat_giong()
T_ve, d_ve = 32, 16
q1, k1 = torch.randn(d_ve, dtype=torch.float64), torch.randn(d_ve, dtype=torch.float64)
pe = hinh_sin(T_ve, d_ve).double()
diem_sin = (q1 + pe) @ (k1 + pe).T                              # vị trí cộng vào nội dung
diem_rope = xoay(q1.expand(T_ve, d_ve)) @ xoay(k1.expand(T_ve, d_ve)).T


def do_lech_cheo(M):
    """Chênh lệch lớn nhất giữa các ô trên cùng một đường chéo."""
    return max((torch.diagonal(M, o).max() - torch.diagonal(M, o).min()).item() for o in range(-T_ve + 1, T_ve))


print(f"hình sin cộng vào: các ô cùng đường chéo lệch nhau tới {do_lech_cheo(diem_sin):.2f}")
print(f"RoPE: các ô cùng đường chéo lệch nhau tới {do_lech_cheo(diem_rope):.1e} (sai số làm tròn FP64)")

fig, axs = plt.subplots(1, 2, figsize=(8.6, 4.0))
for ax, M, ten in ((axs[0], diem_sin, "hình sin cộng vào embedding"), (axs[1], diem_rope, "RoPE: quay q và k")):
    anh = ax.imshow(M.numpy(), cmap="RdBu_r", vmin=-M.abs().max().item(), vmax=M.abs().max().item())
    ax.set_title(ten, fontsize=10)
    ax.set_xlabel("vị trí của khoá n")
    ax.set_ylabel("vị trí của truy vấn m")
    ax.grid(False)
    fig.colorbar(anh, ax=ax, shrink=0.8, label="điểm q·k")
plt.tight_layout()
plt.show()

Bên trái, điểm phụ thuộc cả $m$ lẫn $n$: mô hình phải học riêng cho từng vị trí tuyệt đối. Bên phải, mọi đường chéo
có một màu duy nhất: điểm chỉ phụ thuộc khoảng cách giữa hai token. Phần 7 sẽ cho thấy hệ quả khi chạy mô hình trên
chuỗi dài hơn lúc huấn luyện.

## 5. Nhiều đầu: chia chiều, không thêm phép tính

Tập trung nhiều đầu (multi-head attention, phương trình 8) chạy nhiều hàm tập trung song song, mỗi đầu có bộ phép
chiếu truy vấn, khoá, giá trị riêng, rồi ghép kết quả lại qua $W^O$. Nhưng mỗi đầu chỉ có $d / h$ chiều.

**Dự đoán:** với $d = 512$, đi từ 1 đầu lên 16 đầu thì số tham số và số phép nhân cộng dồn của tầng tập trung tăng
bao nhiêu? Bộ nhớ cho ma trận điểm thì sao?

In [ ]:
d_mh, S_nhieu = 512, sach(4096, nguon="4.096 token · 33,6 MB")
theo_sach("ma trận điểm FP16, một đầu (MB)", S_nhieu * S_nhieu * FP16 / 1e6, sach=33.6, nguon="4.096 token · 33,6 MB")
print(f"{'đầu':>4} {'d mỗi đầu':>10} {'tham số':>12} {'MAC chiếu':>16} {'MAC tập trung':>16} {'điểm FP16 (MB)':>16}")
cac_tham_so = []
for dau in (1, 2, 4, 8, 16):
    lop = nn.MultiheadAttention(d_mh, dau)
    n = sum(p.numel() for p in lop.parameters())
    cac_tham_so.append(n)
    mac_chieu = 4 * S_nhieu * d_mh * d_mh                       # Q, K, V, O
    mac_tap_trung = dau * 2 * S_nhieu * S_nhieu * (d_mh // dau)   # QKᵀ và nhân với V, mỗi đầu
    print(f"{dau:>4} {d_mh // dau:>10} {n:>12,} {mac_chieu:>16,} {mac_tap_trung:>16,} "
          f"{dau * S_nhieu * S_nhieu * FP16 / 1e6:>16,.1f}")
assert len(set(cac_tham_so)) == 1 and cac_tham_so[0] == 4 * d_mh * d_mh + 4 * d_mh
print(f"tham số luôn là 4d² + 4d = {4 * d_mh * d_mh + 4 * d_mh:,}, bất kể số đầu")

Tham số và phép tính không đổi: chia $d$ thành $h$ phần chỉ là cắt cùng một phép chiếu thành nhiều mảnh. Thứ tăng
theo số đầu là số ma trận điểm $S \times S$, mỗi đầu một cái, và đó chính là phần bộ nhớ mà sổ tay 03 đã tính. Đổi
lại, mỗi đầu có softmax riêng, nên có thể nhìn vào một chỗ khác. Phần 6 xem chúng có thực sự làm vậy không.

## 6. Một transformer nhỏ học đảo ngược chuỗi

Bài toán: nhận 16 token (mỗi token là một trong 16 từ), trả ra đúng 16 token đó theo thứ tự ngược. Không có mặt nạ,
vì đầu ra ở vị trí 0 cần token cuối cùng. Mô hình có 2 tầng, mỗi tầng 4 đầu, $d = 64$, khoảng 70 nghìn tham số, huấn
luyện 800 bước với batch 64. Ba biến thể chỉ khác nhau ở cách đưa vị trí vào.

**Dự đoán:** biến thể nào không học được, và vì sao? Nó sẽ đúng được khoảng bao nhiêu phần trăm?

In [ ]:
def bai_dao_nguoc(B, g, T=16):
    x = torch.randint(0, 16, (B, T), generator=g)
    return x, x.flip(1)


kiem_tra_dn = bai_dao_nguoc(1000, torch.Generator().manual_seed(1))
duong_dn, mo_hinh_dn = {}, {}
for vi_tri in ("khong", "sin", "rope"):
    dat_hat_giong()
    mo_hinh_dn[vi_tri] = MoHinh(vi_tri=vi_tri)
    duong_dn[vi_tri], _, giay = huan_luyen(mo_hinh_dn[vi_tri], bai_dao_nguoc, 800, kiem_tra_dn)
    do_tren_may(f"vị trí '{vi_tri}': độ chính xác cuối cùng", round(duong_dn[vi_tri][-1], 3))
    do_tren_may(f"vị trí '{vi_tri}': thời gian huấn luyện", round(giay, 1), "s")

fig, ax = plt.subplots(figsize=(6.8, 3.6))
buoc_ve = np.arange(100, 801, 100)
for (vi_tri, ten), mau in zip((("khong", "không có vị trí"), ("sin", "hình sin"), ("rope", "RoPE")), MAU):
    ax.plot(buoc_ve, duong_dn[vi_tri], "o-", color=mau, label=ten)
ax.axhline(1 / 16, color="0.5", lw=0.8, ls="--")
ax.text(800, 1 / 16 + 0.02, "đoán mò: 1/16", fontsize=7.5, color="0.4", ha="right")
ax.set_ylim(0, 1.05)
ax.set_xlabel("bước huấn luyện")
ax.set_ylabel("độ chính xác trên 1,000 chuỗi kiểm tra")
ax.set_title("Đảo ngược 16 token (đo trên máy này)")
ax.legend(loc="center right", fontsize=8)
plt.tight_layout()
plt.show()

Không có vị trí, mô hình không biết ô đầu ra nào là ô số 0 và ô nào là ô số 15, nên mọi ô thấy cùng một tập token và
chỉ đoán được những gì đúng chung cho cả tập, chẳng hạn token lặp lại nhiều nhất trong chuỗi. Nó hơn đoán mò nhờ vậy,
nhưng không bao giờ học được bài, dù huấn luyện bao lâu. Hai biến thể có vị trí đều học được, RoPE chậm hơn một chút
ở đầu.

Giờ nhìn vào bên trong mô hình hình sin. Ô dưới chạy nó trên 200 chuỗi kiểm tra, lấy trung bình ma trận trọng số của
từng đầu ở từng tầng, và vẽ ra. Hàng là vị trí đầu ra đang hỏi, cột là vị trí đầu vào được nhìn. Nếu một đầu làm đúng
việc đảo ngược, trọng số của nó phải nằm trên đường chéo phụ: vị trí 0 nhìn vị trí 15, vị trí 1 nhìn vị trí 14, ...

In [ ]:
m = mo_hinh_dn["sin"].eval()
with torch.no_grad():
    m(kiem_tra_dn[0][:200])
cheo_phu = torch.arange(16).flip(0)
fig, axs = plt.subplots(2, 4, figsize=(8.8, 4.6), sharex=True, sharey=True, layout="constrained")
for t, khoi in enumerate(m.khoi):
    w = khoi.tap_trung.trong_so.mean(0)                        # (đầu, 16, 16)
    for h in range(4):
        ax = axs[t, h]
        anh = ax.imshow(w[h].numpy(), cmap="Blues", vmin=0, vmax=1)
        ax.set_title(f"tầng {t + 1}, đầu {h + 1}", fontsize=9)
        ax.set_xticks([0, 15])
        ax.set_yticks([0, 15])
        ax.grid(False)
        print(f"tầng {t + 1}, đầu {h + 1}: trọng số trung bình trên đường chéo phụ {w[h, torch.arange(16), cheo_phu].mean():.2f}")
for ax in axs[1]:
    ax.set_xlabel("vị trí được nhìn")
for ax in axs[:, 0]:
    ax.set_ylabel("vị trí hỏi")
fig.colorbar(anh, ax=axs, shrink=0.9, label="trọng số tập trung")
fig.suptitle("Mô hình hình sin sau khi học đảo ngược: trung bình 200 chuỗi (đo trên máy này)", fontsize=10)
plt.show()

Các đầu ở tầng thứ hai thường dồn trọng số lên đường chéo phụ: mỗi vị trí hỏi đúng token nó cần chép. Tầng đầu
thường nhìn rải hơn, chuẩn bị biểu diễn cho tầng sau.

Chương nhận xét rằng các đầu thường dư thừa, nhiều đầu học những khuôn mẫu giống nhau, và đó là lý do cắt tỉa đầu là
một hướng nén mô hình. Ô dưới kiểm điều đó: lần lượt tắt từng đầu (đặt phần tương ứng của $W^O$ về 0) rồi đo lại độ
chính xác.

In [ ]:
with torch.no_grad():
    goc = do_chinh_xac(m, *kiem_tra_dn)
    for t, khoi in enumerate(m.khoi):
        W = khoi.tap_trung.ra.weight
        for h in range(4):
            giu = W[:, h * 16:(h + 1) * 16].clone()
            W[:, h * 16:(h + 1) * 16] = 0                           # tắt đầu h: đầu ra của nó không đi tiếp
            do_tren_may(f"tắt tầng {t + 1}, đầu {h + 1}: độ chính xác", round(do_chinh_xac(m, *kiem_tra_dn), 3))
            W[:, h * 16:(h + 1) * 16] = giu
do_tren_may("không tắt đầu nào: độ chính xác", round(goc, 3))

Nếu tắt một đầu mà độ chính xác gần như không đổi, những đầu còn lại đang làm cùng việc với nó. Ở mô hình nhỏ này,
bốn đầu của tầng hai cùng chép theo đường chéo phụ, nên mất một đầu thì ba đầu kia gánh thay.

## 7. Chuỗi dài hơn lúc huấn luyện

Một mô hình thường được huấn luyện ở một độ dài chuỗi rồi đem dùng ở độ dài khác. Bài toán ở đây: ở mỗi vị trí $i$,
trả ra token ở vị trí $i - 3$ (chép lùi 3 bước), có mặt nạ nhân quả. Huấn luyện ở 16 token, rồi kiểm ở 16, 32 và 64
token. Bài này chỉ cần biết khoảng cách 3, không cần biết vị trí tuyệt đối.

**Dự đoán:** biến thể nào giữ được độ chính xác ở 64 token, khi mọi vị trí từ 16 trở đi chưa từng xuất hiện lúc huấn
luyện?

In [ ]:
DO_TRE = 3


def bai_cham(B, g, T=16):
    x = torch.randint(0, 16, (B, T), generator=g)
    y = torch.cat([torch.full((B, DO_TRE), -100), x[:, :-DO_TRE]], 1)   # đích ở vị trí i là token i − 3
    return x, y


cac_T = [16, 32, 64]
kiem_tra_dai = {T: bai_cham(500, torch.Generator().manual_seed(1), T) for T in cac_T}
ngoai_suy = {}
for vi_tri in ("khong", "sin", "rope"):
    dat_hat_giong()
    m = MoHinh(vi_tri=vi_tri, nhan_qua=True)
    _, _, giay = huan_luyen(m, bai_cham, 600, kiem_tra_dai[16], moi=600)
    ngoai_suy[vi_tri] = [do_chinh_xac(m, *kiem_tra_dai[T]) for T in cac_T]
    for T, a in zip(cac_T, ngoai_suy[vi_tri]):
        do_tren_may(f"vị trí '{vi_tri}', kiểm ở {T} token: độ chính xác", round(a, 3))
    do_tren_may(f"vị trí '{vi_tri}': thời gian huấn luyện", round(giay, 1), "s")

fig, ax = plt.subplots(figsize=(6.8, 3.6))
for (vi_tri, ten), mau in zip((("khong", "không có vị trí"), ("sin", "hình sin"), ("rope", "RoPE")), MAU):
    ax.plot(cac_T, ngoai_suy[vi_tri], "o-", color=mau, label=ten)
ax.axhline(1 / 16, color="0.5", lw=0.8, ls="--", label="đoán mò: 1/16")
ax.set_xscale("log", base=2)
ax.set_xticks(cac_T, [str(T) for T in cac_T])
ax.minorticks_off()
ax.set_ylim(0, 1.05)
ax.set_xlabel("độ dài chuỗi lúc kiểm tra")
ax.set_ylabel("độ chính xác")
ax.set_title("Chép lùi 3 bước, huấn luyện ở 16 token (đo trên máy này)")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

Mã hoá hình sin gắn mỗi vị trí với một vectơ cụ thể, và các vectơ từ vị trí 16 trở đi chưa từng được thấy: mô hình
hỏng ở đúng chỗ đó. Với RoPE, điểm chỉ phụ thuộc khoảng cách, và khoảng cách 3 ở vị trí 50 trông y như khoảng cách 3
ở vị trí 5, nên mô hình giữ được nhiều hơn hẳn, dù không trọn vẹn. Biến thể không có vị trí vẫn hơn đoán mò: mặt nạ
nhân quả tự nó để lộ một chút thông tin vị trí, vì vị trí $i$ chỉ thấy $i + 1$ token.

## Thử thêm

1. Ở phần 2, thay `math.sqrt(d)` bằng `d` trong biến thể có chia. Entropy của hàng softmax đi về đâu khi $d$ lớn?
   Một mô hình có học được gì nếu mọi hàng gần như đều?
2. Ở phần 6, giữ $d = 64$ nhưng tạo mô hình với `dau=1`, rồi `dau=8`. Đường học thay đổi thế nào? Bản đồ tập trung
   của một đầu duy nhất trông ra sao?
3. Ở phần 7, đổi `DO_TRE` thành 8, hoặc truyền `co_so=100.0` vào `xoay`. RoPE còn ngoại suy tốt không?

## Tóm lại

* Tập trung là một phép tra cứu mềm: tích vô hướng đo độ khớp giữa truy vấn và khoá, softmax biến chúng thành trọng
  số có tổng bằng một, và đầu ra là tổ hợp có trọng số của các giá trị.
* Chia cho $\sqrt{d_k}$ đưa phương sai của điểm về 1; không chia thì softmax bão hoà khi đầu rộng ra, và với đầu 128
  chiều thì softmax ngây thơ ở FP16 tràn trên. Trừ giá trị lớn nhất của hàng trước khi lấy mũ là đủ để tránh tràn.
* Mặt nạ nhân quả cho phép huấn luyện mọi vị trí song song mà không vị trí nào thấy tương lai; thiếu nó, mô hình học
  cách đọc đáp án và vô dụng khi sinh token.
* Không có vị trí, tự tập trung đẳng biến với hoán vị. Mã hoá hình sin đưa vị trí tuyệt đối vào nội dung, còn RoPE
  làm điểm chỉ phụ thuộc khoảng cách, nên chịu được chuỗi dài hơn lúc huấn luyện tốt hơn.
* Nhiều đầu không thêm tham số hay phép tính, chỉ thêm ma trận điểm (33.6 MB mỗi đầu ở 4,096 token); các đầu thường
  trùng việc nhau, đúng như chương nhận xét.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kiến trúc mạng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch06-nn-architectures/), dựng từ chương
[*Network Architectures*](https://mlsysbook.ai/vol1/nn_architectures/nn_architectures.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.